# 🔮 AlphaLens — LSTM Trajectory Model Training

**Runtime:** Google Colab GPU (T4/V100)

This notebook trains a dual-head PyTorch LSTM that predicts:
- **5-day price trajectory** (cumulative returns)
- **5-day volatility spread** (upper/lower bounds)

from a 30-day lookback window of 9 engineered technical features.

### Output Artifacts
| File | Description |
|------|-------------|
| `trajectory_lstm.pt` | Model weights (state dict) |
| `scaler_config.json` | RobustScaler center & scale vectors |
| `model_config.json` | Architecture hyperparameters |

In [ ]:
# ─── Cell 1: Install Dependencies ───────────────────────────────────────────
!pip install -q yfinance scikit-learn torch numpy pandas matplotlib

In [ ]:
# ─── Cell 2: Imports & Device Setup ────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ─── Cell 3: Configuration ─────────────────────────────────────────────────

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA", "META", "NFLX"]
LOOKBACK = 30          # Days of historical context
HORIZON = 5            # Days to predict
INPUT_DIM = 9          # Number of technical features
HIDDEN_DIM = 128       # LSTM hidden units
NUM_LAYERS = 2         # LSTM depth
DROPOUT = 0.2
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8           # Early stopping patience
VAL_SPLIT = 0.15

# Output directory (mountable in Colab → download to local machine)
OUTPUT_DIR = Path("/content/alphalens_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Training config: {len(TICKERS)} tickers, {LOOKBACK}→{HORIZON} window")
print(f"🧠 LSTM: input={INPUT_DIM}, hidden={HIDDEN_DIM}, layers={NUM_LAYERS}")
print(f"📂 Artifacts → {OUTPUT_DIR}")

In [ ]:
# ─── Cell 4: Data Ingestion ────────────────────────────────────────────────

def download_data(tickers: list[str], period: str = "5y") -> dict[str, pd.DataFrame]:
    """Download 5 years of daily OHLCV data for each ticker."""
    data = {}
    for ticker in tickers:
        print(f"  ⬇️  Downloading {ticker}...", end=" ")
        df = yf.download(ticker, period=period, interval="1d", progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        if len(df) < LOOKBACK + HORIZON + 60:
            print(f"⚠️ Skipped (only {len(df)} rows)")
            continue
        data[ticker] = df
        print(f"✅ {len(df)} rows")
    return data

print("📡 Downloading 5 years of daily data...")
raw_data = download_data(TICKERS)
print(f"\n✅ Loaded {len(raw_data)} tickers successfully")

In [ ]:
# ─── Cell 5: Feature Engineering ───────────────────────────────────────────

def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Compute RSI and normalize to [-1.0, 1.0] range."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(window=period).mean()
    rs = gain / loss.replace(0, 1e-10)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    # Normalize from [0, 100] → [-1.0, 1.0]
    return (rsi - 50.0) / 50.0


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the 9 technical features per timestep:
      1. Returns:        P_t / P_{t-1} - 1.0
      2. SMA_20_Ratio:   SMA_20(P) / P - 1.0
      3. SMA_50_Ratio:   SMA_50(P) / P - 1.0
      4. BB_Upper_Ratio: (SMA_20 + 2σ_20) / P - 1.0
      5. BB_Lower_Ratio: (SMA_20 - 2σ_20) / P - 1.0
      6. RSI_Norm:        14-day RSI mapped to [-1.0, 1.0]
      7. Volume_Ratio:   V_t / SMA_20(V) - 1.0
      8. High_Low_Ratio: (High - Low) / Close
      9. Open_Close_Ratio: (Close - Open) / Open
    """
    close = df["Close"].copy()
    high = df["High"].copy()
    low = df["Low"].copy()
    open_ = df["Open"].copy()
    volume = df["Volume"].astype(float).copy()

    features = pd.DataFrame(index=df.index)

    # 1. Daily returns
    features["Returns"] = close.pct_change()

    # 2-3. SMA ratios
    sma_20 = close.rolling(window=20).mean()
    sma_50 = close.rolling(window=50).mean()
    features["SMA_20_Ratio"] = sma_20 / close - 1.0
    features["SMA_50_Ratio"] = sma_50 / close - 1.0

    # 4-5. Bollinger Band ratios
    std_20 = close.rolling(window=20).std()
    features["BB_Upper_Ratio"] = (sma_20 + 2.0 * std_20) / close - 1.0
    features["BB_Lower_Ratio"] = (sma_20 - 2.0 * std_20) / close - 1.0

    # 6. RSI normalized
    features["RSI_Norm"] = compute_rsi(close, period=14)

    # 7. Volume ratio
    vol_sma_20 = volume.rolling(window=20).mean()
    features["Volume_Ratio"] = volume / vol_sma_20.replace(0, 1e-10) - 1.0

    # 8. High-Low range ratio
    features["High_Low_Ratio"] = (high - low) / close.replace(0, 1e-10)

    # 9. Open-Close return
    features["Open_Close_Ratio"] = (close - open_) / open_.replace(0, 1e-10)

    return features


# Engineer features for all tickers
all_features = {}
for ticker, df in raw_data.items():
    feats = engineer_features(df)
    # Drop rows with NaN (from rolling windows)
    feats = feats.dropna()
    all_features[ticker] = feats
    print(f"  {ticker}: {len(feats)} feature rows")

print(f"\n✅ Feature engineering complete for {len(all_features)} tickers")

In [ ]:
# ─── Cell 6: Fit RobustScaler & Create Sliding Windows ─────────────────────

FEATURE_COLUMNS = [
    "Returns", "SMA_20_Ratio", "SMA_50_Ratio",
    "BB_Upper_Ratio", "BB_Lower_Ratio", "RSI_Norm",
    "Volume_Ratio", "High_Low_Ratio", "Open_Close_Ratio"
]

# Concatenate all feature data to fit a single global scaler
all_feat_values = pd.concat(
    [feats[FEATURE_COLUMNS] for feats in all_features.values()],
    axis=0
).values

scaler = RobustScaler()
scaler.fit(all_feat_values)
print(f"📐 RobustScaler fitted on {all_feat_values.shape[0]} rows × {all_feat_values.shape[1]} features")
print(f"   Center: {scaler.center_.tolist()}")
print(f"   Scale:  {scaler.scale_.tolist()}")


def create_sliding_windows(
    features_dict: dict[str, pd.DataFrame],
    raw_data_dict: dict[str, pd.DataFrame],
    scaler: RobustScaler,
    lookback: int = 30,
    horizon: int = 5,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Create (X, y_trajectory, y_volatility) sliding window arrays.

    y_trajectory: cumulative returns for the next `horizon` days
    y_volatility: Bollinger-based volatility spread for next `horizon` days
    """
    X_all, y_traj_all, y_vol_all = [], [], []

    for ticker in features_dict:
        feats = features_dict[ticker]
        raw = raw_data_dict[ticker]

        # Align raw close prices with feature index
        close_prices = raw["Close"].loc[feats.index]

        # Scale features
        feat_values = feats[FEATURE_COLUMNS].values
        scaled = scaler.transform(feat_values)

        n = len(scaled)
        for i in range(lookback, n - horizon):
            # Input: lookback window of scaled features
            X_all.append(scaled[i - lookback : i])

            # Target trajectory: cumulative returns from current price
            current_price = close_prices.iloc[i]
            future_prices = close_prices.iloc[i + 1 : i + 1 + horizon]
            if len(future_prices) < horizon:
                continue
            cum_returns = (future_prices.values / current_price) - 1.0
            y_traj_all.append(cum_returns)

            # Target volatility: rolling 20-day std of returns as fraction of price
            bb_upper = feats["BB_Upper_Ratio"].iloc[i + 1 : i + 1 + horizon].values
            bb_lower = feats["BB_Lower_Ratio"].iloc[i + 1 : i + 1 + horizon].values
            # Volatility spread = half the Bollinger width
            vol_spread = np.abs(bb_upper - bb_lower) / 2.0
            if len(vol_spread) < horizon:
                X_all.pop()
                y_traj_all.pop()
                continue
            y_vol_all.append(vol_spread)

    return (
        np.array(X_all, dtype=np.float32),
        np.array(y_traj_all, dtype=np.float32),
        np.array(y_vol_all, dtype=np.float32),
    )


X, y_traj, y_vol = create_sliding_windows(all_features, raw_data, scaler, LOOKBACK, HORIZON)

print(f"\n📊 Dataset created:")
print(f"   X shape:      {X.shape}   (samples × lookback × features)")
print(f"   y_traj shape: {y_traj.shape}  (samples × horizon)")
print(f"   y_vol shape:  {y_vol.shape}  (samples × horizon)")

In [ ]:
# ─── Cell 7: Train/Validation Split & DataLoaders ──────────────────────────

n_samples = len(X)
n_val = int(n_samples * VAL_SPLIT)
n_train = n_samples - n_val

# Temporal split: last VAL_SPLIT% of data is validation (no lookahead bias)
X_train, X_val = X[:n_train], X[n_train:]
y_traj_train, y_traj_val = y_traj[:n_train], y_traj[n_train:]
y_vol_train, y_vol_val = y_vol[:n_train], y_vol[n_train:]

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_traj_train),
    torch.from_numpy(y_vol_train),
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_traj_val),
    torch.from_numpy(y_vol_val),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"📦 Train: {n_train} samples ({len(train_loader)} batches)")
print(f"📦 Val:   {n_val} samples ({len(val_loader)} batches)")

In [ ]:
# ─── Cell 8: Model Definition ──────────────────────────────────────────────

class StockTrajectoryLSTM(nn.Module):
    """
    Dual-head LSTM for stock trajectory + volatility prediction.

    Architecture:
      Input (B, 30, 9) → LSTM(128, 2 layers) → last hidden state
        ├── fc_trajectory → (B, 5) cumulative returns
        └── fc_volatility → (B, 5) strictly positive volatility bounds
    """

    def __init__(
        self,
        input_dim: int = 9,
        hidden_dim: int = 128,
        num_layers: int = 2,
        output_dim: int = 5,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc_trajectory = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, output_dim),
        )
        self.fc_volatility = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, output_dim),
            nn.Softplus(),  # Guarantees strictly positive volatility
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        out, (hn, cn) = self.lstm(x)
        last_hidden = out[:, -1, :]  # (B, hidden_dim)
        trajectory = self.fc_trajectory(last_hidden)
        volatility = self.fc_volatility(last_hidden)
        return trajectory, volatility


model = StockTrajectoryLSTM(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_dim=HORIZON,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🧠 Model on {device}:")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(model)

In [ ]:
# ─── Cell 9: Training Loop ─────────────────────────────────────────────────

traj_criterion = nn.SmoothL1Loss()  # Huber loss for trajectory (robust to outliers)
vol_criterion = nn.MSELoss()        # MSE for volatility
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4, verbose=False
)

# Training history
history = {
    "train_loss": [],
    "val_loss": [],
    "train_traj_loss": [],
    "train_vol_loss": [],
    "val_traj_loss": [],
    "val_vol_loss": [],
    "lr": [],
}

best_val_loss = float("inf")
patience_counter = 0
best_state = None

print(f"🚀 Starting training for {EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_traj_loss_sum = 0.0
    train_vol_loss_sum = 0.0
    train_batches = 0

    for X_batch, y_traj_batch, y_vol_batch in train_loader:
        X_batch = X_batch.to(device)
        y_traj_batch = y_traj_batch.to(device)
        y_vol_batch = y_vol_batch.to(device)

        optimizer.zero_grad()
        pred_traj, pred_vol = model(X_batch)

        loss_traj = traj_criterion(pred_traj, y_traj_batch)
        loss_vol = vol_criterion(pred_vol, y_vol_batch)
        loss = loss_traj + 0.5 * loss_vol  # Weight volatility loss lower

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_traj_loss_sum += loss_traj.item()
        train_vol_loss_sum += loss_vol.item()
        train_batches += 1

    avg_train_traj = train_traj_loss_sum / train_batches
    avg_train_vol = train_vol_loss_sum / train_batches
    avg_train_total = avg_train_traj + 0.5 * avg_train_vol

    # ── Validate ──
    model.eval()
    val_traj_loss_sum = 0.0
    val_vol_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():
        for X_batch, y_traj_batch, y_vol_batch in val_loader:
            X_batch = X_batch.to(device)
            y_traj_batch = y_traj_batch.to(device)
            y_vol_batch = y_vol_batch.to(device)

            pred_traj, pred_vol = model(X_batch)
            loss_traj = traj_criterion(pred_traj, y_traj_batch)
            loss_vol = vol_criterion(pred_vol, y_vol_batch)

            val_traj_loss_sum += loss_traj.item()
            val_vol_loss_sum += loss_vol.item()
            val_batches += 1

    avg_val_traj = val_traj_loss_sum / max(val_batches, 1)
    avg_val_vol = val_vol_loss_sum / max(val_batches, 1)
    avg_val_total = avg_val_traj + 0.5 * avg_val_vol

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(avg_val_total)

    # Record history
    history["train_loss"].append(avg_train_total)
    history["val_loss"].append(avg_val_total)
    history["train_traj_loss"].append(avg_train_traj)
    history["train_vol_loss"].append(avg_train_vol)
    history["val_traj_loss"].append(avg_val_traj)
    history["val_vol_loss"].append(avg_val_vol)
    history["lr"].append(current_lr)

    # Early stopping check
    improved = ""
    if avg_val_total < best_val_loss:
        best_val_loss = avg_val_total
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        improved = " ★ best"
    else:
        patience_counter += 1

    # Log every 5 epochs or on improvement
    if epoch % 5 == 0 or improved or epoch == 1:
        print(
            f"  Epoch {epoch:3d}/{EPOCHS} │ "
            f"Train: {avg_train_total:.6f} (T:{avg_train_traj:.6f} V:{avg_train_vol:.6f}) │ "
            f"Val: {avg_val_total:.6f} (T:{avg_val_traj:.6f} V:{avg_val_vol:.6f}) │ "
            f"LR: {current_lr:.1e}{improved}"
        )

    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

# Restore best weights
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\n✅ Restored best model (val_loss={best_val_loss:.6f})")
else:
    print("\n⚠️ No improvement detected during training")

In [ ]:
# ─── Cell 10: Training Curves Visualization ────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total Loss
axes[0].plot(history["train_loss"], label="Train", color="#10B981", linewidth=2)
axes[0].plot(history["val_loss"], label="Val", color="#EF4444", linewidth=2)
axes[0].set_title("Total Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Trajectory Loss
axes[1].plot(history["train_traj_loss"], label="Train Traj", color="#06B6D4", linewidth=2)
axes[1].plot(history["val_traj_loss"], label="Val Traj", color="#F59E0B", linewidth=2)
axes[1].set_title("Trajectory Loss (SmoothL1)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(history["lr"], color="#8B5CF6", linewidth=2)
axes[2].set_title("Learning Rate Schedule", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Training curves saved.")

In [ ]:
# ─── Cell 11: Export Artifacts ─────────────────────────────────────────────

# 1. Save model weights (CPU state dict for portable inference)
model_cpu = model.cpu()
weights_path = OUTPUT_DIR / "trajectory_lstm.pt"
torch.save(model_cpu.state_dict(), weights_path)
print(f"💾 Model weights saved: {weights_path} ({weights_path.stat().st_size / 1024:.1f} KB)")

# 2. Save scaler configuration
scaler_config = {
    "scaler_type": "RobustScaler",
    "center": scaler.center_.tolist(),
    "scale": scaler.scale_.tolist(),
    "feature_columns": FEATURE_COLUMNS,
}
scaler_path = OUTPUT_DIR / "scaler_config.json"
with open(scaler_path, "w") as f:
    json.dump(scaler_config, f, indent=2)
print(f"💾 Scaler config saved: {scaler_path}")

# 3. Save model configuration
model_config = {
    "model_class": "StockTrajectoryLSTM",
    "input_dim": INPUT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "output_dim": HORIZON,
    "dropout": DROPOUT,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "training": {
        "epochs_completed": len(history["train_loss"]),
        "best_val_loss": float(best_val_loss),
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "tickers": TICKERS,
        "total_samples": n_samples,
        "train_samples": n_train,
        "val_samples": n_val,
    },
}
config_path = OUTPUT_DIR / "model_config.json"
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)
print(f"💾 Model config saved: {config_path}")

print(f"\n✅ All artifacts saved to {OUTPUT_DIR}/")
print(f"   📁 trajectory_lstm.pt")
print(f"   📁 scaler_config.json")
print(f"   📁 model_config.json")
print(f"   📁 training_curves.png")

In [ ]:
# ─── Cell 12: Download Artifacts ───────────────────────────────────────────
# Run this cell to download all 3 files to your local Windows machine.
# Place them in: backend/models/ on your AlphaLens project.

from google.colab import files

print("📥 Downloading artifacts to your local machine...")
print("   Place these files in: AlphaLens/backend/models/\n")

files.download(str(OUTPUT_DIR / "trajectory_lstm.pt"))
files.download(str(OUTPUT_DIR / "scaler_config.json"))
files.download(str(OUTPUT_DIR / "model_config.json"))

print("\n✅ Download complete!")
print("\nNext steps:")
print("  1. Move the 3 files to AlphaLens/backend/models/")
print("  2. Start the backend: uvicorn main:app --reload")
print("  3. Test: GET http://localhost:8000/api/stock-insights/AAPL")

In [ ]:
# ─── Cell 13: Quick Sanity Check — Inference Test ──────────────────────────

# Verify the saved model loads correctly and produces valid output shapes
print("🔬 Running inference sanity check...\n")

# Reload from disk
test_model = StockTrajectoryLSTM(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_dim=HORIZON,
    dropout=DROPOUT,
)
test_model.load_state_dict(torch.load(weights_path, map_location="cpu", weights_only=True))
test_model.eval()

# Create a dummy input
dummy_input = torch.randn(1, LOOKBACK, INPUT_DIM)
with torch.no_grad():
    pred_traj, pred_vol = test_model(dummy_input)

print(f"  Input shape:      {dummy_input.shape}")
print(f"  Trajectory shape: {pred_traj.shape}  values: {pred_traj.squeeze().tolist()}")
print(f"  Volatility shape: {pred_vol.shape}  values: {pred_vol.squeeze().tolist()}")
print(f"  Volatility > 0:   {(pred_vol > 0).all().item()} (Softplus guarantee)")
print(f"\n✅ Sanity check passed!")